# Create and run a local RAG pipeline from scratch

## 1. Document/text processing and embedding creation

### Import PDF Document

In [ ]:
import os
import requests

# Get pdf document path
pdf_path = "human-nutrition-text.pdf"

# Download PDF
if not os.path.exists(pdf_path):
    print(f"[INFO] File does not exist, Downloading ...")
    url = "https://pressbooks.oer.hawaii.edu/humannutrition2/open/download?type=pdf"
    filename=pdf_path
    response = requests.get(url)

    if response.status_code == 200:
        with open(filename, 'wb') as file:
            file.write(response.content)
        print(f"[INFO] Downloaded {filename}")
    else:
        print(f"[ERROR] Failed to download file. Status code: {response.status_code}")

else:
    print(f"[INFO] File already exists, skipping download.")

We've got a pdf, let's open it!

In [ ]:
import fitz  # PyMuPDF
from tqdm.auto import tqdm

def text_formatter(text: str) -> str:
    # Remove newlines and extra spaces
    cleaned_text = text.replace('\n', ' ').strip()

    return cleaned_text

def open_and_read_pdf(pdf_path: str) -> list[dict]:
    doc = fitz.open(pdf_path)
    pages_and_text = []

    # Iterate through each page in the PDF
    for page_num, page in tqdm(enumerate(doc)):
        text = page.get_text()  # Extract text from the page
        text = text_formatter(text)  # Format the extracted text
        
        # Append the formatted text to the list with page number
        pages_and_text.append({
            "page_num": page_num - 41,  # Page numbers are 1-indexed
            "page_char_count": len(text),
            "page_word_count": len(text.split()),
            "page_sentence_count_raw": len(text.split('. ')),
            "page_token_count": len(text.split())/4,
            "text": text
        })

    return pages_and_text

pages_and_text = open_and_read_pdf(pdf_path=pdf_path)
pages_and_text[:3]

In [ ]:
import random

random.sample(pages_and_text, k=3)

In [ ]:
import pandas as pd
df = pd.DataFrame(pages_and_text)
df.head()

In [ ]:
df.describe().round(2)

### Further text processing (splitting pages into sentences)

In [ ]:
from spacy.lang.en import English

nlp = English()
nlp.add_pipe("sentencizer")


In [ ]:
for item in tqdm(pages_and_text):
    doc = nlp(item["text"])
    item["sentences"] = list(doc.sents)

    item["sentences"] = [str(sentence) for sentence in item["sentences"]]
    item["page_sentence_count_spacy"] = len(item["sentences"])

In [ ]:
random.sample(pages_and_text, k=3)

In [ ]:
df=pd.DataFrame(pages_and_text)
df.describe().round(2)

### Chunking our sentence together

In [ ]:
num_sentence_chunk_size = 10

def split_list(input_list: list[str], 
               slice_size: int=num_sentence_chunk_size) -> list[list[str]]:
    return [input_list[i:i + slice_size] for i in range(0, len(input_list), slice_size)]

test_list=list(range(23))
split_list(test_list)

In [ ]:
for item in tqdm(pages_and_text):
    doc = nlp(item["text"])
    item["sentence_chunks"] = split_list(input_list=item["sentences"], 
                                         slice_size=num_sentence_chunk_size)
    item["num_chunks"] = len(item["sentence_chunks"])

In [ ]:
random.sample(pages_and_text, k=1)

### Splitting each chunk into its own item

In [ ]:
import re

pages_and_chunks = []
for item in tqdm(pages_and_text):
    for sentence_chunk in item["sentence_chunks"]:
        chunk_dict = {}
        chunk_dict["page_num"] = item["page_num"]

        joined_sentence_chunk = " ".join(sentence_chunk).replace("  "," ").strip()
        joined_sentence_chunk = re.sub(r'\.(A-Z)', r'. \1', joined_sentence_chunk)

        chunk_dict["sentence_chunk"] = joined_sentence_chunk

        chunk_dict["chunk_char_count"] = len(joined_sentence_chunk)
        chunk_dict["chunk_word_count"] = len([word for word in joined_sentence_chunk.split(" ")])
        chunk_dict["chunk_token_count"] = len(joined_sentence_chunk)/4
        pages_and_chunks.append(chunk_dict)

len(pages_and_chunks)

In [ ]:
df=pd.DataFrame(pages_and_chunks)
df.describe().round(2)

### Filter chunks of text for short chunks

In [ ]:
min_token_length=30
pages_and_chunks_over_min_token_len = df[df["chunk_token_count"] > min_token_length].to_dict(orient="records")
pages_and_chunks_over_min_token_len[:3]

### Embedding our text chunks

In [ ]:
from sentence_transformers import SentenceTransformer
sentences = ["This is an example sentence", "Each sentence is converted"]

embedding_model = SentenceTransformer(model_name_or_path='sentence-transformers/all-mpnet-base-v2', 
                            device="cpu")
embeddings = embedding_model.encode(sentences)
print(embeddings)

In [ ]:
print(embeddings.shape)

In [ ]:
%%time

embedding_model.to("cuda")

for item in tqdm(pages_and_chunks_over_min_token_len):
    item["embedding"] = embedding_model.encode(item["sentence_chunk"])

In [ ]:
%%time

text_chunks=[item["sentence_chunk"] for item in pages_and_chunks_over_min_token_len]
text_chunks[419]

In [ ]:
len(text_chunks)

In [ ]:
%%time

text_chunks_embeddings = embedding_model.encode(text_chunks,
                                               batch_size=32,
                                               convert_to_tensor=True)

text_chunks_embeddings

### Save embeddings to file

In [ ]:
pages_and_chunks_over_min_token_len[419]

In [ ]:
text_chunks_and_embeddings_df = pd.DataFrame(pages_and_chunks_over_min_token_len)
embeddings_df_save_path = "text_chunks_and_embeddings_df.csv"
text_chunks_and_embeddings_df.to_csv(embeddings_df_save_path, index=False)

In [ ]:
text_chunks_and_embedding_df_load=pd.read_csv(embeddings_df_save_path)
text_chunks_and_embedding_df_load.head()

## 2. RAG - Search and Answer

In [ ]:
import random

import torch
import numpy as np
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"

text_chunks_and_embedding_df = pd.read_csv("text_chunks_and_embeddings_df.csv")

text_chunks_and_embedding_df["embedding"] = text_chunks_and_embedding_df["embedding"].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))

embeddings = torch.tensor(np.stack(text_chunks_and_embedding_df["embedding"].tolist(), axis=0), dtype=torch.float32).to(device)

pages_and_chunks = text_chunks_and_embedding_df.to_dict(orient="records")

text_chunks_and_embedding_df

In [ ]:
embeddings.shape

In [ ]:
from sentence_transformers import util, SentenceTransformer

embedding_model = SentenceTransformer(model_name_or_path='all-mpnet-base-v2', 
                            device=device)

In [ ]:
embeddings.shape

In [ ]:
query = "good foods for protein"
print(f"Query: {query}")

query_embedding = embedding_model.encode(query, convert_to_tensor=True).to("cuda")

from time import perf_counter as timer
start_time = timer()
dot_scores = util.dot_score(a=query_embedding, b=embeddings)[0]
end_time = timer()

print(f"[INFO]Time taken to get scores on {len(embeddings)} embeddings : {end_time - start_time:.5f} seconds.")

top_results_dot_product = torch.topk(dot_scores, k=5)
top_results_dot_product

In [ ]:
larger_embeddings = torch.randn(100*embeddings.shape[0], 768).to(device)
print(f"Shape of larger embeddings: {larger_embeddings.shape}")

start_time = timer()
dot_scores = util.dot_score(a=query_embedding, b=larger_embeddings)[0]
end_time = timer()

print(f"[INFO]Time taken to get scores on {len(larger_embeddings)} embeddings : {end_time - start_time:.5f} seconds.")


In [ ]:
import textwrap

def print_wrapped(text, wrap_length=80):
    wrapped_text = textwrap.fill(text, width=wrap_length)
    print(wrapped_text)

In [ ]:
query = "good foods for protein"
print(f"Query : '{query}'\n")
print("Results:")

for score, idx in zip(top_results_dot_product[0], top_results_dot_product[1]):
    print(f"Score: {score:.4f}")
    print("Text:")
    print_wrapped(pages_and_chunks[idx]["sentence_chunk"])
    print(f"Page Number: {pages_and_chunks[idx]['page_num']}")
    print("\n")

In [ ]:
import fitz

pdf_path = "human-nutrition-text.pdf"
doc = fitz.open(pdf_path)
page = doc.load_page(411 + 41)

img = page.get_pixmap(dpi=300)

doc.close()

img_array = np.frombuffer(img.samples_mv, 
                          dtype=np.uint8).reshape(img.h, img.w, img.n)

import matplotlib.pyplot as plt
plt.figure(figsize=(13, 10))
plt.imshow(img_array)
plt.title(f"Query: '{query}' | Most Relevant Page:")
plt.axis("off")
plt.show()

### Similarity measures: dot product and cosine similarity

In [ ]:
import torch

def dot_product(vector1, vector2):
    return torch.dot(vector1, vector2)

def cosine_similarity(vector1, vector2):
    dot_product = torch.dot(vector1, vector2)

    norm_vector1 = torch.sqrt(torch.sum(vector1**2))
    norm_vector2 = torch.sqrt(torch.sum(vector2**2))

    return dot_product / (norm_vector1 * norm_vector2)

vector1 = torch.tensor([1, 2, 3], dtype=torch.float32)
vector2 = torch.tensor([1, 2, 3], dtype=torch.float32)
vector3 = torch.tensor([4, 5, 6], dtype=torch.float32)
vector4 = torch.tensor([-1, -2, -3], dtype=torch.float32)

print(f"Dot Product of vector1 and vector2: {dot_product(vector1, vector2):.4f}")
print(f"Dot Product of vector1 and vector3: {dot_product(vector1, vector3):.4f}")
print(f"Dot Product of vector1 and vector4: {dot_product(vector1, vector4):.4f}")

print(f"Cosine Similarity of vector1 and vector2: {cosine_similarity(vector1, vector2):.4f}")
print(f"Cosine Similarity of vector1 and vector3: {cosine_similarity(vector1, vector3):.4f}")
print(f"Cosine Similarity of vector1 and vector4: {cosine_similarity(vector1, vector4):.4f}")

### Functionizing our semantic search pipeline

In [ ]:
def retrieve_relevant_resources(query: str, 
                             embeddings: torch.Tensor, 
                             model: SentenceTransformer=embedding_model, 
                             n_resources_to_return: int=5,
                             print_time: bool=True):
    
    query_embedding = model.encode(query, convert_to_tensor=True)

    start_time=timer()
    dot_scores = util.dot_score(query_embedding, embeddings)[0]
    end_time=timer()
    
    if print_time:
        print(f"[INFO]Time taken to get scores on {len(embeddings)} embeddings : {end_time - start_time:.5f} seconds.")

        scores,indices = torch.topk(input=dot_scores, k=n_resources_to_return)
    
    return scores, indices

def print_top_results_and_scores(query: str, 
                                 embeddings: torch.Tensor,
                                 pages_and_chunks: list[dict]=pages_and_chunks, 
                                 n_resources_to_return: int=5):
    
    scores, indices = retrieve_relevant_resources(query=query, 
                                                  embeddings=embeddings, 
                                                  n_resources_to_return=n_resources_to_return)
    
    for score, idx in zip(scores, indices):
        print(f"Score: {score:.4f}")
        print("Text:")
        print_wrapped(pages_and_chunks[idx]["sentence_chunk"])
        print(f"Page Number: {pages_and_chunks[idx]['page_num']}")
        print("\n")

In [ ]:
query = "foods high in fiber"
#retrieve_relevant_resources(query=query, embeddings=embeddings)
print_top_results_and_scores(query=query, embeddings=embeddings)

### Getting an LLM for local generation

#### Checking our local GPU memory availability

In [ ]:
import torch
gpu_memory_bytes = torch.cuda.get_device_properties(0).total_memory
gpu_memory_gb = round(gpu_memory_bytes / (2 ** 30))
print(f"GPU Memory: {gpu_memory_gb} GB")

In [ ]:
# Note: the following is Gemma focused, however, there are more and more LLMs of the 2B and 7B size appearing for local use.
if gpu_memory_gb < 5.1:
    print(f"Your available GPU memory is {gpu_memory_gb}GB, you may not have enough memory to run a Gemma LLM locally without quantization.")
elif gpu_memory_gb < 8.1:
    print(f"GPU memory: {gpu_memory_gb} | Recommended model: Gemma 2B in 4-bit precision.")
    use_quantization_config = True 
    model_id = "google/gemma-2b-it"
elif gpu_memory_gb < 19.0:
    print(f"GPU memory: {gpu_memory_gb} | Recommended model: Gemma 2B in float16 or Gemma 7B in 4-bit precision.")
    use_quantization_config = False 
    model_id = "google/gemma-2b-it"
elif gpu_memory_gb > 19.0:
    print(f"GPU memory: {gpu_memory_gb} | Recommend model: Gemma 7B in 4-bit or float16 precision.")
    use_quantization_config = False 
    model_id = "google/gemma-7b-it"

print(f"use_quantization_config set to: {use_quantization_config}")
print(f"model_id set to: {model_id}")

### Loading an LLM locally

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers.utils import is_flash_attn_2_available 

torch.cuda.empty_cache()

# 1. Configuration for your 6GB GPU
# We use 4-bit quantization to leave room for your context (textbook data)
model_id = "google/gemma-2b-it" 

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4", # Highly recommended for better quality
    bnb_4bit_use_double_quant=True
)

# 2. Attention Setup
# If you haven't run 'pip install flash-attn', this will automatically use 'sdpa'
# if is_flash_attn_2_available() and torch.cuda.get_device_capability(0)[0] >= 8:
#     attn_implementation = "flash_attention_2"
# else:
#     attn_implementation = "sdpa" 

print(f"[INFO] Using model: {model_id}")
# print(f"[INFO] Attention: {attn_implementation}")

# device_map = {"": 0}

# 3. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 4. Load Model
# device_map="auto" is the secret sauce for your 6GB limit
llm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    low_cpu_mem_usage=True,
    device_map = "auto"
    # attn_implementation=attn_implementation
)

print("Gemma 2B is ready!")

In [ ]:
def get_model_num_params(model: torch.nn.Module):
    return sum([p.numel() for p in model.parameters()])

get_model_num_params(llm_model)

### Generate text with our LLM

In [ ]:
input_text = "What are the macronutrients and what are their functions in the body?"
print(f"Input text:\n{input_text}")

dialog_template = [
    {"role": "user", "content": input_text}
]

prompt=tokenizer.apply_chat_template(conversation=dialog_template,
                                     tokenize=False,
                                     add_generation_prompt=True)
print(f"Prompt after applying chat template:\n{prompt}")

In [ ]:
%%time

input_ids = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = llm_model.generate(**input_ids,
                            max_new_tokens=256)
print(f"Model output tokens:\n{outputs[0]}\n")

In [ ]:
outputs_decoded = tokenizer.decode(outputs[0])
print(f"Decoded output:\n{outputs_decoded}\n")

In [ ]:
# Nutrition-style questions generated with GPT4
gpt4_questions = [
    "What are the macronutrients, and what roles do they play in the human body?",
    "How do vitamins and minerals differ in their roles and importance for health?",
    "Describe the process of digestion and absorption of nutrients in the human body.",
    "What role does fibre play in digestion? Name five fibre containing foods.",
    "Explain the concept of energy balance and its importance in weight management."
]

# Manually created question list
manual_questions = [
    "How often should infants be breastfed?",
    "What are symptoms of pellagra?",
    "How does saliva help with digestion?",
    "What is the RDI for protein per day?",
    "water soluble vitamins"
]

query_list = gpt4_questions + manual_questions
query_list

In [ ]:
import random

query = random.choice(query_list)
print(f"Query: {query}")
scores, indices = retrieve_relevant_resources(query=query, embeddings=embeddings)
scores, indices

### Augmenting our prompt with context items

In [ ]:
pages_and_chunks[420]

In [ ]:
def prompt_formatter(query: str, 
                     context_items: list[dict]) -> str:
    context = "- " + "\n- ".join([item["sentence_chunk"] for item in context_items])
    base_prompt = """Based on the following context items, please answer the query.
Give yourself room to think by extracting relevant passages from the context before answering the query.
Don't return the thinking, only return the answer.
Make sure your answers are as explanatory as possible.
Use the following examples as reference for the ideal answer style.
\nExample 1:
Query: What are the fat-soluble vitamins?
Answer: The fat-soluble vitamins include Vitamin A, Vitamin D, Vitamin E, and Vitamin K. These vitamins are absorbed along with fats in the diet and can be stored in the body's fatty tissue and liver for later use. Vitamin A is important for vision, immune function, and skin health. Vitamin D plays a critical role in calcium absorption and bone health. Vitamin E acts as an antioxidant, protecting cells from damage. Vitamin K is essential for blood clotting and bone metabolism.
\nExample 2:
Query: What are the causes of type 2 diabetes?
Answer: Type 2 diabetes is often associated with overnutrition, particularly the overconsumption of calories leading to obesity. Factors include a diet high in refined sugars and saturated fats, which can lead to insulin resistance, a condition where the body's cells do not respond effectively to insulin. Over time, the pancreas cannot produce enough insulin to manage blood sugar levels, resulting in type 2 diabetes. Additionally, excessive caloric intake without sufficient physical activity exacerbates the risk by promoting weight gain and fat accumulation, particularly around the abdomen, further contributing to insulin resistance.
\nExample 3:
Query: What is the importance of hydration for physical performance?
Answer: Hydration is crucial for physical performance because water plays key roles in maintaining blood volume, regulating body temperature, and ensuring the transport of nutrients and oxygen to cells. Adequate hydration is essential for optimal muscle function, endurance, and recovery. Dehydration can lead to decreased performance, fatigue, and increased risk of heat-related illnesses, such as heat stroke. Drinking sufficient water before, during, and after exercise helps ensure peak physical performance and recovery.
\nNow use the following context items to answer the user query:
{context}
\nRelevant passages: <extract relevant passages from the context here>
User query: {query}
Answer:"""
    # Update base prompt with context items and query   
    base_prompt = base_prompt.format(context=context, query=query)

    # Create prompt template for instruction-tuned model
    dialogue_template = [
        {"role": "user",
        "content": base_prompt}
    ]

    # Apply the chat template
    prompt = tokenizer.apply_chat_template(conversation=dialogue_template,
                                          tokenize=False,
                                          add_generation_prompt=True)
    return prompt

query = random.choice(query_list)
print(f"Query: {query}")

scores, indices = retrieve_relevant_resources(query=query, embeddings=embeddings)

context_items = [pages_and_chunks[i] for i in indices]

prompt = prompt_formatter(query=query, context_items=context_items)
print(prompt)

In [ ]:
%%time

input_ids = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = llm_model.generate(**input_ids,
                            temperature=0.7,
                            do_sample=True,
                            max_new_tokens=256)

output_text = tokenizer.decode(outputs[0])
print(f"Query: {query}")
print(f"RAG answer: \m{output_text.replace(prompt, '')}")

### Functionize our LLM answering feature

In [ ]:
def ask(query: str,
        temperature: float=0.7,
        max_new_tokens: int=256,
        format_answer_text=True,
        return_answer_only=True):
    """
    Takes a user query, retrieves relevant context from the textbook, and generates an answer using the LLM.
    """
    sources, indices = retrieve_relevant_resources(query=query, embeddings=embeddings)

    context_items = [pages_and_chunks[i] for i in indices]
    for i, item in enumerate(context_items):
        item["score"] = sources[i].cpu()

    prompt = prompt_formatter(query=query, context_items=context_items)

    input_ids = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = llm_model.generate(**input_ids,
                                temperature=temperature,
                                do_sample=True,
                                max_new_tokens=max_new_tokens)

    output_text = tokenizer.decode(outputs[0])

    if format_answer_text:
        output_text = output_text.replace(prompt, "").replace("<bos>", "").replace("<eos>", "")

    if return_answer_only:
        return output_text

    return output_text, context_items 

In [ ]:
query = random.choice(query_list)
print(f"Query: {query}\n")
ask(query=query,
    temperature=0.3,
    return_answer_only=False)

In [ ]:
# ChromaDB + BM25 hybrid retrieval + cross-encoder reranker
# Install extras if you need them (uncomment to run in a fresh env)
# !pip install chromadb rank_bm25

from chromadb import Client
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
import numpy as np

# Prepare text chunks and ids (uses `pages_and_chunks_over_min_token_len` built earlier)
text_chunks = [item["sentence_chunk"] for item in pages_and_chunks_over_min_token_len]
metadatas = [{"page_num": item.get("page_num")} for item in pages_and_chunks_over_min_token_len]
ids = [f"chunk_{i}" for i in range(len(text_chunks))]

# 1) Compute / ensure embeddings are available as numpy arrays
try:
    # try to reuse already-computed embeddings if present
    embeddings_np = np.vstack([np.array(item["embedding"]) if not isinstance(item["embedding"], str) else np.fromstring(item["embedding"].strip("[]"), sep=" ") for item in pages_and_chunks_over_min_token_len])
except Exception:
    # fallback: compute with existing `embedding_model`
    embeddings_np = embedding_model.encode(text_chunks, batch_size=32, convert_to_numpy=True)

# 2) Create / populate ChromaDB collection
client = Client()
try:
    collection = client.get_collection(name="text_chunks")
except Exception:
    collection = client.create_collection(name="text_chunks")

# If collection empty, add documents (safe to call multiple times; we check length)
existing = 0
try:
    existing = len(collection.get(ids=[ids[0]])["ids"]) if len(ids) else 0
except Exception:
    existing = 0

if existing == 0:
    collection.add(ids=ids, documents=text_chunks, metadatas=metadatas, embeddings=embeddings_np.tolist())

# 3) Build BM25 index for lexical matching
tokenized = [doc.split() for doc in text_chunks]
bm25 = BM25Okapi(tokenized)

# 4) Cross-encoder reranker
device_for_reranker = "cuda" if torch.cuda.is_available() else "cpu"
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=device_for_reranker)

def normalize_scores(x: np.ndarray) -> np.ndarray:
    if x.size == 0:
        return x
    mn, mx = x.min(), x.max()
    if mx == mn:
        return np.zeros_like(x)
    return (x - mn) / (mx - mn)

def hybrid_retrieval(query: str, top_k: int = 50, alpha: float = 0.5):
    """Return candidate passages from hybrid BM25 + vector similarity (alpha weights BM25)."""
    # BM25 scores
    query_tokens = query.split()
    bm25_scores = np.array(bm25.get_scores(query_tokens))

    # Vector scores (dot product)
    query_emb = embedding_model.encode(query, convert_to_numpy=True)
    vec_scores = embeddings_np.dot(query_emb)

    # Normalize and combine
    bm25_n = normalize_scores(bm25_scores)
    vec_n = normalize_scores(vec_scores)
    combined = alpha * bm25_n + (1 - alpha) * vec_n

    top_idx = np.argsort(-combined)[:top_k]

    candidates = []
    for i in top_idx:
        candidates.append({
            "id": ids[i],
            "text": text_chunks[i],
            "page_num": metadatas[i].get("page_num"),
            "bm25_score": float(bm25_scores[i]),
            "vec_score": float(vec_scores[i]),
            "combined_score": float(combined[i])
        })
    return candidates

def rerank_with_cross_encoder(query: str, candidates: list[dict], top_k: int = 5):
    """Rerank candidate passages with the cross-encoder and return top_k items."""
    pairs = [(query, c["text"]) for c in candidates]
    scores = reranker.predict(pairs)
    for c, s in zip(candidates, scores):
        c["rerank_score"] = float(s)
    candidates = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)
    return candidates[:top_k]

# Example usage
query = "good foods for protein"
print(f"Query: {query}\n")
candidates = hybrid_retrieval(query=query, top_k=50, alpha=0.4)
top5 = rerank_with_cross_encoder(query=query, candidates=candidates, top_k=5)

for i, item in enumerate(top5, 1):
    print(f"Rank {i}: rerank_score={item['rerank_score']:.4f} | combined={item['combined_score']:.4f} | page={item['page_num']}")
    print(item["text"][:400])
    print("---\n")